# Athena emulator — AWS CLI ops

You administer this Athena footprint from a shell: the AWS CLI is your
interface, and every call is `aws athena … --endpoint-url` pointed at the
emulator. Each section runs the commands the CLI's own examples ship
(`aws athena <op> help`) against the live stack — sample names swapped
for this run's throwaway resources — and records a measured row per
capability. Where the doc text diverges from what the CLI or engine
accepts, the cell runs the corrected form and says so.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls
must still land on moto.

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
# A pager would eat the stdout the %%bash cells capture for assertion.
os.environ["AWS_PAGER"] = ""

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import json
import shutil
import sys
import uuid
from pathlib import Path

import boto3
from _evidence import persist, record, render
from _helpers import (
    delete_run_buckets,
    delete_run_catalogs,
    delete_run_databases,
    delete_workgroup_contents,
    object_keys_under,
    run_workgroup_names,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL, (
    athena.meta.endpoint_url
)
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL, s3.meta.endpoint_url
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL, glue.meta.endpoint_url

# Kernels spawned by JupyterLab inherit the container PATH, not the venv's —
# put the resolved `aws` binary's directory on PATH so the %%bash cells read
# `aws athena …` verbatim.
aws_bin = Path(sys.executable).parent / "aws"
if not aws_bin.is_file():
    found = shutil.which("aws")
    assert found is not None, (
        "aws CLI not installed next to sys.executable or on PATH"
    )
    aws_bin = Path(found)
os.environ["PATH"] = f"{aws_bin.parent}:{os.environ['PATH']}"

REGION = os.environ["AWS_DEFAULT_REGION"]
ACCOUNT_ID = "123456789012"  # accepted but not enforced by the emulator
RUN_SUFFIX = uuid.uuid4().hex[:8]
# Every resource this run creates carries the nb07- prefix (nb07_ for Glue
# databases, where '-' is invalid) so cleanup can discover leftovers — its
# own and any failed run's — instead of tracking them in a list.
ATHENA_RESOURCE_PREFIX = "nb07-"
DATABASE_PREFIX = "nb07_"

bucket = f"{ATHENA_RESOURCE_PREFIX}{uuid.uuid4().hex[:12]}"
database = f"{DATABASE_PREFIX}logs_{RUN_SUFFIX}"
meta_database = f"{DATABASE_PREFIX}sample_{RUN_SUFFIX}"
geo_database = f"{DATABASE_PREFIX}geo_{RUN_SUFFIX}"
workgroup = f"{ATHENA_RESOURCE_PREFIX}wg-{RUN_SUFFIX}"
team_workgroup = f"{ATHENA_RESOURCE_PREFIX}team-{RUN_SUFFIX}"
catalog = f"{ATHENA_RESOURCE_PREFIX}dynamo-{RUN_SUFFIX}"
cw_catalog = f"{ATHENA_RESOURCE_PREFIX}cw-{RUN_SUFFIX}"
unused_catalog = f"{ATHENA_RESOURCE_PREFIX}unused-{RUN_SUFFIX}"

# %%bash interpolates ${name}/{name} from the Python namespace, but only if
# every field resolves — one miss leaves the whole cell literal. The NB07_*
# (and AWS_*) names below exist only in the environment, never as Python
# variables, so every cell stays literal for bash to expand.
os.environ.update(
    {
        "NB07_ACCOUNT": ACCOUNT_ID,
        "NB07_BUCKET": bucket,
        "NB07_CAT": catalog,
        "NB07_CAT_ARN": f"arn:aws:athena:{REGION}:{ACCOUNT_ID}:datacatalog/{catalog}",
        "NB07_CAT_CW": cw_catalog,
        "NB07_CAT_UNUSED": unused_catalog,
        "NB07_DB": database,
        "NB07_GEO_DB": geo_database,
        "NB07_META_DB": meta_database,
        "NB07_WG": workgroup,
        "NB07_WG_ARN": f"arn:aws:athena:{REGION}:{ACCOUNT_ID}:workgroup/{workgroup}",
        "NB07_WG_TEAM": team_workgroup,
    }
)

s3.create_bucket(Bucket=bucket)


def cli_blocks(out: str) -> dict[str, object]:
    """Split a ``--out`` transcript on ``=== name`` markers into parsed JSON.

    A block whose payload isn't JSON (``key=value`` lines, plain text) comes
    back as its raw string — marker names keep every command's output
    addressable no matter how many share one cell.
    """
    blocks: dict[str, object] = {}
    for chunk in out.split("=== ")[1:]:
        name, _, payload = chunk.partition("\n")
        text = payload.strip()
        blocks[name.strip()] = (
            json.loads(text) if text.startswith(("{", "[")) else text
        )
    return blocks


def kv_lines(text: object) -> dict[str, str]:
    """Parse a captured ``key=value`` marker block into a dict."""
    return dict(
        line.split("=", 1) for line in str(text).splitlines() if "=" in line
    )


# Bound by ``%%bash --out`` as each capture cell runs; predeclared so the
# notebook's static checks see the names before their consumer cells do.
out_smoke = out_wg = out_wgtags = out_cat = out_meta = ""
out_nqc = out_nq = out_q1 = out_q2 = out_q3 = out_q4 = ""

print(f"aws → {aws_bin}")
print(f"workgroup={workgroup} catalog={catalog} bucket={bucket}")

aws → /home/gabriel-menezes/Documents/repos/mlops/.venv/bin/aws
workgroup=nb07-wg-c9d7ca70 catalog=nb07-dynamo-c9d7ca70 bucket=nb07-027ae2a26aff


## The CLI, pointed at the emulator

`aws --version` pins the binary (awscli 1.46.1 — the dev-group pin in
`libs/athena-local`); `--endpoint-url` on every call routes Athena traffic
to the emulator while S3/Glue keep their own endpoint.

In [3]:
%%bash --out out_smoke
set -euo pipefail
echo "=== version"
aws --version 2>&1
echo "=== workgroups"
aws athena list-work-groups \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [4]:
blocks = cli_blocks(out_smoke)
assert "aws-cli/1.46.1" in str(blocks["version"]), blocks["version"]
listed = {entry["Name"] for entry in blocks["workgroups"]["WorkGroups"]}
assert "primary" in listed

record(
    "aws binary + --endpoint-url routing (awscli 1.46.1)",
    "PASS",
    f"{aws_bin} → {ATHENA_URL}; list-work-groups → {len(listed)} group(s) "
    "incl. primary",
)
blocks["workgroups"]

{'WorkGroups': [{'Name': 'primary',
   'State': 'ENABLED',
   'Description': '',
   'CreationTime': 1791930100.8710053,
   'EngineVersion': {'SelectedEngineVersion': 'AUTO',
    'EffectiveEngineVersion': 'Athena engine version 3'}}]}

## Workgroups

`create-work-group` carries the result-location configuration and
create-time tags; `update-work-group` flips state in place; the spare
group exits through `delete-work-group`.

In [5]:
%%bash
set -euo pipefail
aws athena create-work-group \
    --name "$NB07_WG" \
    --configuration ResultConfiguration={OutputLocation="s3://$NB07_BUCKET/results/"},EnforceWorkGroupConfiguration="true",PublishCloudWatchMetricsEnabled="true" \
    --description "Workgroup for data analysts" \
    --tags Key=Division,Value=West Key=Location,Value=Seattle Key=Team,Value="Big Data" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

aws athena create-work-group \
    --name "$NB07_WG_TEAM" \
    --description "cli-example seed" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [6]:
%%bash --out out_wg
set -euo pipefail
echo "=== get"
aws athena get-work-group \
    --work-group "$NB07_WG" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== update-team"
aws athena update-work-group \
    --work-group "$NB07_WG_TEAM" \
    --state DISABLED \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== get-team"
aws athena get-work-group \
    --work-group "$NB07_WG_TEAM" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== delete-team"
aws athena delete-work-group \
    --work-group "$NB07_WG_TEAM" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== list"
aws athena list-work-groups \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [7]:
blocks = cli_blocks(out_wg)
created = blocks["get"]["WorkGroup"]
assert created["Name"] == workgroup
assert created["State"] == "ENABLED"
assert created["Configuration"]["EnforceWorkGroupConfiguration"] is True
assert created["Configuration"]["ResultConfiguration"]["OutputLocation"] == (
    f"s3://{bucket}/results/"
)

disabled = blocks["get-team"]["WorkGroup"]
assert disabled["Name"] == team_workgroup
assert disabled["State"] == "DISABLED"

listed = {entry["Name"] for entry in blocks["list"]["WorkGroups"]}
assert {"primary", workgroup} <= listed
assert team_workgroup not in listed

record(
    "create/get-work-group — configuration + create-time tags applied",
    "PASS",
    f"{workgroup} ENABLED, OutputLocation=s3://{bucket}/results/",
)
record(
    "update-work-group --state DISABLED + delete-work-group",
    "PASS",
    f"{team_workgroup} DISABLED on read-back; absent from list after delete",
)

Evidence(feature='update-work-group --state DISABLED + delete-work-group', status='PASS', detail='nb07-team-c9d7ca70 DISABLED on read-back; absent from list after delete', latency_ms=None)

## Tags on a workgroup

`create-work-group --tags` already stamped three pairs; the doc's
`list-tags-for-resource` reads them back off the workgroup ARN.

In [8]:
%%bash --out out_wgtags
set -euo pipefail
echo "=== tags"
aws athena list-tags-for-resource \
    --resource-arn "$NB07_WG_ARN" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [9]:
tags = {
    entry["Key"]: entry["Value"]
    for entry in cli_blocks(out_wgtags)["tags"]["Tags"]
}
assert tags == {
    "Division": "West",
    "Location": "Seattle",
    "Team": "Big Data",
}

record(
    "list-tags-for-resource on a workgroup ARN",
    "PASS",
    f"{len(tags)} create-time tag(s): {sorted(tags)}",
)

Evidence(feature='list-tags-for-resource on a workgroup ARN', status='PASS', detail="3 create-time tag(s): ['Division', 'Location', 'Team']", latency_ms=None)

## Data catalogs

Register a LAMBDA catalog, correct its function ARN with
`update-data-catalog`, tag it for cost allocation. The doc's
`--function=` flag is rejected by the CLI itself — the model carries only
`Parameters` — so the update runs its corrected `--parameters function=…`
form.

In [10]:
%%bash
set -euo pipefail
aws athena create-data-catalog \
    --name "$NB07_CAT" \
    --type LAMBDA \
    --description "DynamoDB Catalog" \
    --parameters "function=arn:aws:lambda:$AWS_DEFAULT_REGION:$NB07_ACCOUNT:function:dynamo_db_lambda" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
aws athena create-data-catalog \
    --name "$NB07_CAT_CW" \
    --type LAMBDA \
    --description "CloudWatch Logs Catalog" \
    --parameters "function=arn:aws:lambda:$AWS_DEFAULT_REGION:$NB07_ACCOUNT:function:cw_logs_lambda" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
aws athena create-data-catalog \
    --name "$NB07_CAT_UNUSED" \
    --type LAMBDA \
    --description "cli-example seed" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

{
    "DataCatalog": {
        "Name": "nb07-dynamo-c9d7ca70",
        "Description": "DynamoDB Cata

log",
        "Type": "LAMBDA",
        "Parameters": {
            "function": "arn:aws:lambda:us-e

ast-1:123456789012:function:dynamo_db_lambda",
            "catalog": "nb07-dynamo-c9d7ca70",
      

      "metadata-function": "arn:aws:lambda:us-east-1:123456789012:function:dynamo_db_lambda",
      

      "record-function": "arn:aws:lambda:us-east-1:123456789012:function:dynamo_db_lambda"
        }

...

 "record-function": "arn:aws:lambda:us-east-1:123456789012:function:cw_logs_lambda"
        }
    }


}


{
    "DataCatalog": {
        "Name": "nb07-unused-c9d7ca70",
        "Description": "cli-example s

eed",
        "Type": "LAMBDA",
        "Parameters": {
            "catalog": "nb07-unused-c9d7ca70

"
        }
    }
}


In [11]:
%%bash --out out_cat
set -euo pipefail
echo "=== get"
aws athena get-data-catalog \
    --name "$NB07_CAT" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== update-cw"
aws athena update-data-catalog \
    --name "$NB07_CAT_CW" \
    --type LAMBDA \
    --description "New CloudWatch Logs Catalog" \
    --parameters "function=arn:aws:lambda:$AWS_DEFAULT_REGION:$NB07_ACCOUNT:function:new_cw_logs_lambda" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== get-cw"
aws athena get-data-catalog \
    --name "$NB07_CAT_CW" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== tag"
aws athena tag-resource \
    --resource-arn "$NB07_CAT_ARN" \
    --tags Key=Organization,Value=Retail Key=Division,Value=Mountain Key=Product_Line,Value=Shoes Key=Location,Value=Denver \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== tags"
aws athena list-tags-for-resource \
    --resource-arn "$NB07_CAT_ARN" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== untag"
aws athena untag-resource \
    --resource-arn "$NB07_CAT_ARN" \
    --tag-keys Specialization Focus \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== tags-after"
aws athena list-tags-for-resource \
    --resource-arn "$NB07_CAT_ARN" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== delete-unused"
aws athena delete-data-catalog \
    --name "$NB07_CAT_UNUSED" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== list"
aws athena list-data-catalogs \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [12]:
blocks = cli_blocks(out_cat)
dynamo = blocks["get"]["DataCatalog"]
assert dynamo["Name"] == catalog
assert dynamo["Type"] == "LAMBDA"
assert dynamo["Parameters"]["function"].endswith(":function:dynamo_db_lambda")

updated = blocks["get-cw"]["DataCatalog"]
assert updated["Description"] == "New CloudWatch Logs Catalog"
assert updated["Parameters"]["function"].endswith(
    ":function:new_cw_logs_lambda"
)

tags = {entry["Key"]: entry["Value"] for entry in blocks["tags"]["Tags"]}
assert tags == {
    "Organization": "Retail",
    "Division": "Mountain",
    "Product_Line": "Shoes",
    "Location": "Denver",
}
# untag removes keys whether or not they were ever set — Specialization and
# Focus were absent to begin with, the four create-call tags all remain.
assert {entry["Key"] for entry in blocks["tags-after"]["Tags"]} == set(tags)

catalogs = {
    entry["CatalogName"]: entry["Type"]
    for entry in blocks["list"]["DataCatalogsSummary"]
}
assert catalogs["AwsDataCatalog"] == "GLUE"
assert catalogs[catalog] == catalogs[cw_catalog] == "LAMBDA"
assert unused_catalog not in catalogs

record(
    "create/get/list/delete-data-catalog (LAMBDA + built-in GLUE)",
    "PASS",
    f"{catalog} LAMBDA alongside AwsDataCatalog GLUE; {unused_catalog} deleted",
)
record(
    "update-data-catalog via corrected --parameters form",
    "PASS",
    "doc --function flag is CLI-rejected; --parameters function=… rewrote "
    "description + function ARN",
)
record(
    "tag-resource / untag-resource / list-tags-for-resource on a catalog ARN",
    "PASS",
    f"4 tags applied, {len(tags)} remain after removing two unset keys",
)

Evidence(feature='tag-resource / untag-resource / list-tags-for-resource on a catalog ARN', status='PASS', detail='4 tags applied, 4 remain after removing two unset keys', latency_ms=None)

## Catalog metadata

`list-databases`/`get-database`/`list-table-metadata`/`get-table-metadata`
read through `AwsDataCatalog` into Glue — seed two databases and a table
each so the listings have something to show.

In [13]:
for name in (meta_database, geo_database):
    glue.create_database(DatabaseInput={"Name": name})
glue.create_table(
    DatabaseName=meta_database,
    TableInput={
        "Name": "counties",
        "TableType": "EXTERNAL_TABLE",
        "Parameters": {"EXTERNAL": "TRUE"},
    },
)
glue.create_table(
    DatabaseName=geo_database,
    TableInput={
        "Name": "regions",
        "TableType": "EXTERNAL_TABLE",
        "Parameters": {"EXTERNAL": "TRUE"},
    },
)
print(f"seeded: {meta_database}.counties, {geo_database}.regions")

seeded: nb07_sample_c9d7ca70.counties, nb07_geo_c9d7ca70.regions


In [14]:
%%bash --out out_meta
set -euo pipefail
echo "=== dbs"
aws athena list-databases \
    --catalog-name AwsDataCatalog \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== db"
aws athena get-database \
    --catalog-name AwsDataCatalog \
    --database-name "$NB07_META_DB" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== tables"
aws athena list-table-metadata \
    --catalog-name AwsDataCatalog \
    --database-name "$NB07_GEO_DB" \
    --max-items 2 \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== table"
aws athena get-table-metadata \
    --catalog-name AwsDataCatalog \
    --database-name "$NB07_META_DB" \
    --table-name counties \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [15]:
blocks = cli_blocks(out_meta)
databases = {entry["Name"] for entry in blocks["dbs"]["DatabaseList"]}
assert {meta_database, geo_database} <= databases
assert blocks["db"]["Database"]["Name"] == meta_database

tables = {entry["Name"] for entry in blocks["tables"]["TableMetadataList"]}
assert "regions" in tables
assert blocks["table"]["TableMetadata"]["Name"] == "counties"

record(
    "list-databases / get-database read the Glue catalog",
    "PASS",
    f"{meta_database} + {geo_database} in DatabaseList",
)
record(
    "list-table-metadata / get-table-metadata",
    "PASS",
    f"{geo_database}.regions listed; {meta_database}.counties fetched",
)

Evidence(feature='list-table-metadata / get-table-metadata', status='PASS', detail='nb07_geo_c9d7ca70.regions listed; nb07_sample_c9d7ca70.counties fetched', latency_ms=None)

## Named queries

Saved queries live in a workgroup: `create-named-query` returns the id
the rest of the chain (get → list → batch-get → delete) hangs off.

In [16]:
%%bash --out out_nqc
set -euo pipefail
aws athena create-named-query \
    --name "SEA to JFK delayed flights Jan 2016" \
    --description "Both arrival and departure delayed more than 10 minutes." \
    --database "$NB07_META_DB" \
    --query-string "SELECT flightdate, carrier, flightnum, origin, dest, depdelayminutes, arrdelayminutes FROM $NB07_META_DB.flights_parquet WHERE yr = 2016 AND month = 1 AND origin = '\"SEA\"' AND dest = '\"JFK\"' AND depdelayminutes > 10 AND arrdelayminutes > 10" \
    --work-group "$NB07_WG" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [17]:
named_query_id = json.loads(out_nqc)["NamedQueryId"]
os.environ["NB07_NQID"] = named_query_id
named_query_id

'e97c35c0-5772-43c3-a17c-3824b26ed97d'

In [18]:
%%bash --out out_nq
set -euo pipefail
echo "=== get"
aws athena get-named-query \
    --named-query-id "$NB07_NQID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== list"
aws athena list-named-queries \
    --work-group "$NB07_WG" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== batch"
aws athena batch-get-named-query \
    --named-query-ids "$NB07_NQID" a1b2c3d4-5678-90ab-cdef-EXAMPLE22222 a1b2c3d4-5678-90ab-cdef-EXAMPLE33333 \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== delete"
aws athena delete-named-query \
    --named-query-id "$NB07_NQID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== list-after"
aws athena list-named-queries \
    --work-group "$NB07_WG" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"

In [19]:
blocks = cli_blocks(out_nq)
named = blocks["get"]["NamedQuery"]
assert named["Name"] == "SEA to JFK delayed flights Jan 2016"
assert named["Database"] == meta_database
assert named["WorkGroup"] == workgroup

assert named_query_id in blocks["list"]["NamedQueryIds"]
batch = blocks["batch"]
assert [q["NamedQueryId"] for q in batch["NamedQueries"]] == [named_query_id]
assert {e["NamedQueryId"] for e in batch["UnprocessedNamedQueryIds"]} == {
    "a1b2c3d4-5678-90ab-cdef-EXAMPLE22222",
    "a1b2c3d4-5678-90ab-cdef-EXAMPLE33333",
}
assert named_query_id not in blocks["list-after"]["NamedQueryIds"]

record(
    "create/get/list/delete-named-query under the workgroup",
    "PASS",
    f"{named_query_id[:8]}… created in {workgroup}, fetched, listed, deleted",
)
record(
    "batch-get-named-query splits found vs unprocessed ids",
    "PASS",
    "1 found + 2 unprocessed sample ids",
)

Evidence(feature='batch-get-named-query splits found vs unprocessed ids', status='PASS', detail='1 found + 2 unprocessed sample ids', latency_ms=None)

## Running queries

`start-query-execution` submits and hands back an id;
`get-query-execution` polls to a terminal state; `get-query-results`
reads the rows. First the doc's DDL variant creates this run's database.

In [20]:
%%bash --out out_q1
set -euo pipefail
DDL_ID=$(aws athena start-query-execution \
    --query-string "create database if not exists $NB07_DB" \
    --work-group "$NB07_WG" \
    --query QueryExecutionId --output text \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
for _ in $(seq 1 60); do
    STATE=$(aws athena get-query-execution \
        --query-execution-id "$DDL_ID" \
        --query 'QueryExecution.Status.State' --output text \
        --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
    case "$STATE" in SUCCEEDED|FAILED|CANCELLED) break ;; esac
    sleep 1
done
[ "$STATE" = "SUCCEEDED" ] || { echo "ddl never terminal: $STATE" >&2; exit 1; }
echo "=== ddl"
aws athena get-query-execution \
    --query-execution-id "$DDL_ID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== ids"
echo "ddl=$DDL_ID"

In [21]:
blocks = cli_blocks(out_q1)
ddl_execution = blocks["ddl"]["QueryExecution"]
ddl_id = kv_lines(blocks["ids"])["ddl"]

assert ddl_execution["Status"]["State"] == "SUCCEEDED"
assert ddl_execution["StatementType"] == "DDL"
ddl_location = ddl_execution["ResultConfiguration"]["OutputLocation"]
assert ddl_location.endswith(f"{ddl_id}.txt"), ddl_location

record(
    "start-query-execution DDL (create database) → SUCCEEDED",
    "PASS",
    f"{ddl_id[:8]}… → {ddl_location}",
)

Evidence(feature='start-query-execution DDL (create database) → SUCCEEDED', status='PASS', detail='02571481… → s3://nb07-027ae2a26aff/results/02571481-68cc-42d7-b213-8d5e47b6c3c9.txt', latency_ms=None)

A CTAS seed builds the doc's `cloudfront_logs` table, then the
documented SELECT runs against it — with `status = '200'` quoted: the
doc's bare `200` compares a varchar column to an integer, which Trino
rejects.

In [22]:
%%bash --out out_q2
set -euo pipefail
SEED_ID=$(aws athena start-query-execution \
    --query-string "CREATE TABLE cloudfront_logs WITH (external_location = 's3://$NB07_BUCKET/cloudfront_logs/', format = 'PARQUET') AS SELECT * FROM (VALUES ('2023-01-01', 'SFO', 'GET', 'Chrome/1', '/index.html', '200'), ('2023-01-02', 'JFK', 'POST', 'Safari/1', '/login', '403')) AS t(date, location, method, browser, uri, status)" \
    --work-group "$NB07_WG" \
    --query-execution-context "Database=$NB07_DB,Catalog=AwsDataCatalog" \
    --query QueryExecutionId --output text \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
for _ in $(seq 1 90); do
    STATE=$(aws athena get-query-execution \
        --query-execution-id "$SEED_ID" \
        --query 'QueryExecution.Status.State' --output text \
        --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
    case "$STATE" in SUCCEEDED|FAILED|CANCELLED) break ;; esac
    sleep 1
done
[ "$STATE" = "SUCCEEDED" ] || { echo "seed never terminal: $STATE" >&2; exit 1; }

START_NS=$(date +%s%N)
DML_ID=$(aws athena start-query-execution \
    --query-string "select date, location, browser, uri, status from cloudfront_logs where method = 'GET' and status = '200' and location like 'SFO%' limit 10" \
    --work-group "$NB07_WG" \
    --query-execution-context "Database=$NB07_DB,Catalog=AwsDataCatalog" \
    --query QueryExecutionId --output text \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
for _ in $(seq 1 90); do
    STATE=$(aws athena get-query-execution \
        --query-execution-id "$DML_ID" \
        --query 'QueryExecution.Status.State' --output text \
        --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
    case "$STATE" in SUCCEEDED|FAILED|CANCELLED) break ;; esac
    sleep 1
done
END_NS=$(date +%s%N)
[ "$STATE" = "SUCCEEDED" ] || { echo "dml never terminal: $STATE" >&2; exit 1; }

echo "=== dml"
aws athena get-query-execution \
    --query-execution-id "$DML_ID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== results"
aws athena get-query-results \
    --query-execution-id "$DML_ID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== ids"
echo "seed=$SEED_ID"
echo "dml=$DML_ID"
echo "dml_ns=$((END_NS - START_NS))"

In [23]:
blocks = cli_blocks(out_q2)
ids = kv_lines(blocks["ids"])
dml_id = ids["dml"]
os.environ["NB07_DML_ID"] = dml_id

dml_execution = blocks["dml"]["QueryExecution"]
assert dml_execution["Status"]["State"] == "SUCCEEDED"
assert dml_execution["StatementType"] == "DML"
dml_location = dml_execution["ResultConfiguration"]["OutputLocation"]
assert dml_location.endswith(f"{dml_id}.csv"), dml_location
assert f"results/{dml_id}.csv" in object_keys_under(s3, bucket, "results/")

rows = [
    [cell.get("VarCharValue") for cell in row["Data"]]
    for row in blocks["results"]["ResultSet"]["Rows"]
]
assert rows[0] == ["date", "location", "browser", "uri", "status"]
assert rows[1] == ["2023-01-01", "SFO", "Chrome/1", "/index.html", "200"]

record(
    "CTAS seed via start-query-execution (DDL → Glue + parquet)",
    "PASS",
    f"cloudfront_logs staged at s3://{bucket}/cloudfront_logs/",
)
record(
    "start → get-query-execution poll → .csv artifact (DML)",
    "PASS",
    f"{dml_id[:8]}… SUCCEEDED; results/{dml_id}.csv on moto",
    latency_ms=float(ids["dml_ns"]) / 1e6,
)
record(
    "get-query-results returns header row + data rows",
    "PASS",
    f"{len(rows)} row(s), first={rows[1][:2]}",
)

Evidence(feature='get-query-results returns header row + data rows', status='PASS', detail="2 row(s), first=['2023-01-01', 'SFO']", latency_ms=None)

The doc's third `start-query-execution` variant — `CREATE OR REPLACE
VIEW` — names no workgroup, which would land on the shared `primary`;
passing the run workgroup keeps the deploy-time default untouched.
`list-query-executions` and `batch-get-query-execution` close the loop.

In [24]:
%%bash --out out_q3
set -euo pipefail
VIEW_ID=$(aws athena start-query-execution \
    --query-string "CREATE OR REPLACE VIEW cf10 AS SELECT * FROM cloudfront_logs limit 10" \
    --work-group "$NB07_WG" \
    --query-execution-context "Database=$NB07_DB" \
    --query QueryExecutionId --output text \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
for _ in $(seq 1 60); do
    STATE=$(aws athena get-query-execution \
        --query-execution-id "$VIEW_ID" \
        --query 'QueryExecution.Status.State' --output text \
        --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
    case "$STATE" in SUCCEEDED|FAILED|CANCELLED) break ;; esac
    sleep 1
done
[ "$STATE" = "SUCCEEDED" ] || { echo "view never terminal: $STATE" >&2; exit 1; }

echo "=== executions"
aws athena list-query-executions \
    --work-group "$NB07_WG" \
    --max-items 10 \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== batch"
aws athena batch-get-query-execution \
    --query-execution-ids "$NB07_DML_ID" a1b2c3d4-5678-90ab-cdef-EXAMPLE22222 \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
echo "=== ids"
echo "view=$VIEW_ID"

In [25]:
blocks = cli_blocks(out_q3)
ids = kv_lines(blocks["ids"])
listed = set(blocks["executions"]["QueryExecutionIds"])
assert {ddl_id, ids["view"], dml_id} <= listed

batch = blocks["batch"]
assert [q["QueryExecutionId"] for q in batch["QueryExecutions"]] == [dml_id]
assert [
    e["QueryExecutionId"] for e in batch["UnprocessedQueryExecutionIds"]
] == ["a1b2c3d4-5678-90ab-cdef-EXAMPLE22222"]

record(
    "start-query-execution DDL variant (CREATE OR REPLACE VIEW)",
    "PASS",
    f"cf10 built under {workgroup} — shared primary untouched",
)
record(
    "list-query-executions + batch-get-query-execution",
    "PASS",
    f"{len(listed)} execution id(s) in workgroup; batch → 1 found + 1 "
    "unprocessed",
)

Evidence(feature='list-query-executions + batch-get-query-execution', status='PASS', detail='4 execution id(s) in workgroup; batch → 1 found + 1 unprocessed', latency_ms=None)

## Cancelling a query

`stop-query-execution` lands before the engine finishes: three capped
`unnest` cross-joins (~1e12 probes) can't complete inside the round-trip,
so the execution ends `CANCELLED`.

In [26]:
%%bash --out out_q4
set -euo pipefail
SLOW_ID=$(aws athena start-query-execution \
    --query-string "SELECT count(*) FROM unnest(sequence(1, 10000)) a CROSS JOIN unnest(sequence(1, 10000)) b CROSS JOIN unnest(sequence(1, 10000)) c" \
    --work-group "$NB07_WG" \
    --query QueryExecutionId --output text \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
aws athena stop-query-execution \
    --query-execution-id "$SLOW_ID" \
    --endpoint-url "$AWS_ENDPOINT_URL_ATHENA"
for _ in $(seq 1 60); do
    STATE=$(aws athena get-query-execution \
        --query-execution-id "$SLOW_ID" \
        --query 'QueryExecution.Status.State' --output text \
        --endpoint-url "$AWS_ENDPOINT_URL_ATHENA")
    case "$STATE" in SUCCEEDED|FAILED|CANCELLED) break ;; esac
    sleep 1
done
echo "=== ids"
echo "slow=$SLOW_ID"
echo "state=$STATE"

In [27]:
stopped = kv_lines(cli_blocks(out_q4)["ids"])
assert stopped["state"] == "CANCELLED", stopped

record(
    "stop-query-execution cancels a running query",
    "PASS",
    f"{stopped['slow'][:8]}… → CANCELLED",
)

Evidence(feature='stop-query-execution cancels a running query', status='PASS', detail='0891ec1b… → CANCELLED', latency_ms=None)

## Capabilities demonstrated

Every feature above recorded a measured row. The matrix persists to
`parity/07_aws_cli.md`; `PARITY.md` at the project root is regenerated
from all fragments.

In [28]:
print(render())
persist("07_aws_cli")

| feature | status | detail | latency_ms |
|---|---|---|---|
| aws binary + --endpoint-url routing (awscli 1.46.1) | PASS | /home/gabriel-menezes/Documents/repos/mlops/.venv/bin/aws → http://localhost:5001; list-work-groups → 1 group(s) incl. primary |  |
| create/get-work-group — configuration + create-time tags applied | PASS | nb07-wg-c9d7ca70 ENABLED, OutputLocation=s3://nb07-027ae2a26aff/results/ |  |
| update-work-group --state DISABLED + delete-work-group | PASS | nb07-team-c9d7ca70 DISABLED on read-back; absent from list after delete |  |
| list-tags-for-resource on a workgroup ARN | PASS | 3 create-time tag(s): ['Division', 'Location', 'Team'] |  |
| create/get/list/delete-data-catalog (LAMBDA + built-in GLUE) | PASS | nb07-dynamo-c9d7ca70 LAMBDA alongside AwsDataCatalog GLUE; nb07-unused-c9d7ca70 deleted |  |
| update-data-catalog via corrected --parameters form | PASS | doc --function flag is CLI-rejected; --parameters function=… rewrote description + function ARN |  |
| tag

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/07_aws_cli.md')

## Clean up

The run's workgroups (with their named queries), catalogs, bucket, and
Glue databases are throwaway — delete them so the stack is left as you
found it.

In [29]:
for name in run_workgroup_names(athena, ATHENA_RESOURCE_PREFIX):
    delete_workgroup_contents(athena, name)
    athena.delete_work_group(WorkGroup=name)
delete_run_catalogs(athena, ATHENA_RESOURCE_PREFIX)
delete_run_buckets(s3, ATHENA_RESOURCE_PREFIX)
delete_run_databases(glue, DATABASE_PREFIX)
print("cleaned: nb07 workgroups + catalogs + buckets + databases removed")

cleaned: nb07 workgroups + catalogs + buckets + databases removed
